In [ ]:
import torch
import torchvision.models as models

# 1. Load the pre-trained AlexNet with ImageNet weights
# Using the modern 'weights' parameter (replaces 'pretrained=True')
alexnet = models.alexnet(weights='AlexNet_Weights.IMAGENET1K_V1')

# 2. Set the model to evaluation mode
alexnet.eval()

# 3. Inspect the 'features' part of AlexNet to identify the layers
print(alexnet.features)

In [ ]:
# Extract up to the ReLU layer after the 5th Convolution
# index 12 is Conv2d(256, 256, kernel_size=(3, 3)...)
# index 13 is ReLU(inplace=True)
features_extractor = alexnet.features[:12]  # This will include layers up to Conv5

print("Feature extractor defined up to Conv5 + ReLU.")

In [ ]:
# Create a dummy input of 227x227 (Batch size=1, Channels=3, H=227, W=227)
dummy_input = torch.randn(1, 3, 227, 227)

# Forward pass through our extractor
with torch.no_grad(): # Disable gradient calculation for efficiency
    features_output = features_extractor(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {features_output.shape}") 
# Expected Output: torch.Size([1, 256, 13, 13])

In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.io as sio
from torchvision import models, transforms
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from PIL import Image

In [2]:
# Force using CPU
device = torch.device("cpu")

# Load AlexNet
alexnet = models.alexnet(weights='IMAGENET1K_V1').to(device).eval()

# CORRECTED INDEX: Extract up to Conv5 + ReLU (Index 11)
# Slice [:12] gets layers 0 through 11
features_extractor = nn.Sequential(*list(alexnet.features)[:12])

def extract_and_resize_features(img_path):
    """
    Extract Conv5+ReLU features and interpolate to 227x227 using CPU
    """
    preprocess = transforms.Compose([
        transforms.Resize((227, 227)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    img = Image.open(img_path).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Feature shape: [1, 256, 13, 13]
        raw_features = features_extractor(input_tensor)
        
        # Interpolate to match input resolution 227x227
        interpolated = F.interpolate(raw_features, size=(227, 227), mode='bilinear', align_corners=False)
        
    return interpolated.squeeze(0).numpy() # (256, 227, 227)



In [ ]:
# --- Configuration for 1:59 loop ---
pic_dir = 'pics'
target_dir = 'human_map_resized' # The folder containing .npy files
n_features = 256
n_images = 59

all_X = []
all_Y = []

print(f"Starting weights training for {n_images} images...")

for num in range(1, n_images + 1):
    # 1. Construct file paths based on your new naming convention
    img_path = os.path.join(pic_dir, f"testv2_{num}.jpg")
    # Matching your .npy files: human_avg_map_num_resized.npy
    target_path = os.path.join(target_dir, f"human_avg_map_{num}_resized.npy")
    
    if os.path.exists(img_path) and os.path.exists(target_path):
        # 2. Extract Conv5+ReLU features
        # features_output is [1, 256, 13, 13], we interpolate to 227x227
        features = extract_and_resize_features(img_path) # (256, 227, 227)
        
        # 3. Load ground truth from .npy file
        target_map = np.load(target_path)
        
        # 4. Memory-efficient subsampling (every 4th pixel)
        # Using [::4, ::4] to prevent CPU memory overload
        x_flat = features[:, ::4, ::4].reshape(n_features, -1).T
        y_flat = target_map[::4, ::4].flatten()
        
        all_X.append(x_flat)
        all_Y.append(y_flat)
    else:
        print(f"Warning: Missing file for index {num}")

# --- Data Aggregation ---
# Stack all subsampled pixels into large matrices
X_final = np.concatenate(all_X, axis=0) 
Y_final = np.concatenate(all_Y, axis=0)




Starting weights training for 59 images...
